In [4]:
import numpy as np

# =============================================================================
# --- 1. DATASET SESSION NORMALE : 32 ÉTUDIANTS ---
# =============================================================================
# Colonnes : [Heures Étude,Participation %,Résultat]
# Résultat : 1=Réussite / 0=Échec

data=np.array([
[8,90,1],[7,85,1],[6,80,1],[5,70,1],
[4,65,1],[3,60,0],[2,40,0],[1,20,0],
[9,95,1],[8,88,1],[7,75,1],[6,72,1],
[5,68,1],[4,55,0],[3,50,0],[2,35,0],
[10,98,1],[9,92,1],[8,86,1],[7,78,1],
[6,74,1],[5,66,1],[4,58,0],[3,45,0],
[2,30,0],[1,15,0],[0,10,0],[6,70,1],
[7,82,1],[5,62,1],[3,48,0],[2,25,0]
])

# Séparation des variables
X=data[:,:2]
y=data[:,2]

# =============================================================================
# --- 2. CLASSE NAIVE BAYES GAUSSIEN ---
# =============================================================================
class MyNaiveBayes:
    def __init__(self):
        self.classes=None
        self.mean=None
        self.var=None
        self.priors=None

    def fit(self,X,y):

        self.classes=np.unique(y)
        n_features=X.shape[1]

        self.mean=np.zeros((len(self.classes),n_features))
        self.var=np.zeros((len(self.classes),n_features))
        self.priors=np.zeros(len(self.classes))

        for idx,c in enumerate(self.classes):

            X_c=X[y==c]

            self.mean[idx,:]=X_c.mean(axis=0)
            self.var[idx,:]=X_c.var(axis=0)+1e-6
            self.priors[idx]=X_c.shape[0]/X.shape[0]

    def gaussian_density(self,class_idx,x):

        mean=self.mean[class_idx]
        var=self.var[class_idx]

        numerator=np.exp(-((x-mean)**2)/(2*var))
        denominator=np.sqrt(2*np.pi*var)

        return numerator/denominator

    def predict_proba(self,X):

        probabilities=[]

        for x in X:

            posteriors=[]

            for idx,c in enumerate(self.classes):

                prior=np.log(self.priors[idx])

                conditional=np.sum(
                np.log(self.gaussian_density(idx,x))
                )

                posterior=prior+conditional
                posteriors.append(posterior)

            posteriors=np.exp(posteriors-np.max(posteriors))
            probs=posteriors/posteriors.sum()

            probabilities.append(probs)

        return np.array(probabilities)

    def predict(self,X):

        probs=self.predict_proba(X)

        return np.argmax(probs,axis=1)

# =============================================================================
# --- 3. ENTRAÎNEMENT DU MODÈLE ---
# =============================================================================
model=MyNaiveBayes()
model.fit(X,y)

# =============================================================================
# --- 4. PRÉDICTIONS ---
# =============================================================================
predictions=model.predict(X)
probabilities=model.predict_proba(X)

# =============================================================================
# --- 5. AFFICHAGE DES ÉTUDIANTS ---
# =============================================================================
print("========== ANALYSE NAIVE BAYES ==========\n")

for i in range(len(X)):

    heures=X[i][0]
    participation=X[i][1]

    proba_echec=probabilities[i][0]*100
    proba_reussite=probabilities[i][1]*100

    prediction=predictions[i]

    if prediction==1:
        resultat="RÉUSSITE"
    else:
        resultat="ÉCHEC"

    print(f"Étudiant {i+1}")
    print(f"Heures d'étude : {heures}h")
    print(f"Participation : {participation}%")
    print(f"Probabilité Échec : {proba_echec:.2f}%")
    print(f"Probabilité Réussite : {proba_reussite:.2f}%")
    print(f"Décision Finale : {resultat}")
    print("----------------------------------------")

# =============================================================================
# --- 6. TABLEAU STATISTIQUE ---
# =============================================================================
print("\n========== TABLEAU STATISTIQUE ==========\n")

# Variables
heures=X[:,0]
participation=X[:,1]

# Moyennes
moyenne_heures=np.mean(heures)
moyenne_participation=np.mean(participation)

# Médianes
mediane_heures=np.median(heures)
mediane_participation=np.median(participation)

# Variances
variance_heures=np.var(heures)
variance_participation=np.var(participation)

# Écart-types
ecart_heures=np.std(heures)
ecart_participation=np.std(participation)

# Covariance
covariance_matrix=np.cov(heures,participation)

# Affichage tableau
print("------------------------------------------------------------")
print("Mesure                 Heures Étude     Participation")
print("------------------------------------------------------------")
print(f"Moyenne                {moyenne_heures:.2f}              {moyenne_participation:.2f}")
print(f"Médiane                {mediane_heures:.2f}              {mediane_participation:.2f}")
print(f"Variance               {variance_heures:.2f}              {variance_participation:.2f}")
print(f"Écart-Type             {ecart_heures:.2f}              {ecart_participation:.2f}")
print("------------------------------------------------------------")

# =============================================================================
# --- 7. MATRICE DE COVARIANCE ---
# =============================================================================
print("\n========== MATRICE DE COVARIANCE ==========\n")

print(covariance_matrix)

# =============================================================================
# --- 8. PARAMÈTRES DU MODÈLE ---
# =============================================================================
print("\n========== PARAMÈTRES DU MODÈLE ==========\n")

for idx,c in enumerate(model.classes):

    if c==1:
        classe_name="RÉUSSITE"
    else:
        classe_name="ÉCHEC"

    print(f"Classe : {classe_name}")
    print(f"Probabilité Initiale : {model.priors[idx]*100:.2f}%")
    print(f"Moyenne Heures Étude : {model.mean[idx][0]:.2f}")
    print(f"Moyenne Participation : {model.mean[idx][1]:.2f}")
    print(f"Variance Heures Étude : {model.var[idx][0]:.2f}")
    print(f"Variance Participation : {model.var[idx][1]:.2f}")
    print("----------------------------------------")

# =============================================================================
# --- 9. STATISTIQUES GLOBALES ---
# =============================================================================
total_reussite=np.sum(predictions==1)
total_echec=np.sum(predictions==0)

print("\n========== STATISTIQUES GLOBALES ==========\n")

print(f"Nombre Total Étudiants : {len(X)}")
print(f"Nombre Réussites : {total_reussite}")
print(f"Nombre Échecs : {total_echec}")
print(f"Taux Réussite : {(total_reussite/len(X))*100:.2f}%")
print(f"Taux Échec : {(total_echec/len(X))*100:.2f}%")

========== ANALYSE NAIVE BAYES ==========

Étudiant 1
Heures d'étude : 8h
Participation : 90%
Probabilité Échec : 0.00%
Probabilité Réussite : 100.00%
Décision Finale : RÉUSSITE
----------------------------------------
Étudiant 2
Heures d'étude : 7h
Participation : 85%
Probabilité Échec : 0.00%
Probabilité Réussite : 100.00%
Décision Finale : RÉUSSITE
----------------------------------------
Étudiant 3
Heures d'étude : 6h
Participation : 80%
Probabilité Échec : 0.01%
Probabilité Réussite : 99.99%
Décision Finale : RÉUSSITE
----------------------------------------
Étudiant 4
Heures d'étude : 5h
Participation : 70%
Probabilité Échec : 1.28%
Probabilité Réussite : 98.72%
Décision Finale : RÉUSSITE
----------------------------------------
Étudiant 5
Heures d'étude : 4h
Participation : 65%
Probabilité Échec : 33.74%
Probabilité Réussite : 66.26%
Décision Finale : RÉUSSITE
----------------------------------------
Étudiant 6
Heures d'étude : 3h
Participation : 60%
Probabilité Échec : 93.99%
P